# Fuga de clientes Telco: desarrollo en Google Colab

Este notebook reproduce en Colab el flujo de desarrollo del proyecto: revisión del dataset, entrenamiento con `train.py`, pruebas y la app de Streamlit con un túnel temporal.

El despliegue final no se hace aquí, sino en una VM de Google Cloud con Docker (ver `README.md`).

> El almacenamiento de Colab es temporal. Todo lo que importa vive en el repositorio de GitHub.

## 1. Clonar el repositorio e instalar dependencias

In [ ]:
!git clone https://github.com/melo11miguel/telco-churn-streamlit.git
%cd telco-churn-streamlit
!pip -q install -r requirements-dev.txt

## 2. Revisión mínima del dataset

Antes de modelar respondemos las preguntas de la guía:

- **¿Qué representa una fila?** Un cliente de una compañía de telecomunicaciones.
- **¿Cuál es el objetivo?** Predecir `Churn` (Yes/No): si el cliente canceló el servicio.
- **¿Qué variables pueden entrar al modelo?** Las que se conocen de un cliente activo: antigüedad, cargo mensual, contrato, servicios contratados y forma de pago.
- **¿Hay datos faltantes?** Solo en `TotalCharges` (11 vacíos), que además no usamos.
- **¿Hay numéricas y categóricas?** Sí: 3 numéricas y el resto categóricas.

In [ ]:
import pandas as pd

df = pd.read_csv("data/telco_churn.csv")
print(df.shape)
display(df.head())
display(df.dtypes)

In [ ]:
# TotalCharges viene como texto: los vacíos solo aparecen al convertirlo a número.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
display(df.isna().sum().sort_values(ascending=False).head(15))
display(df.describe(include="all").T)

In [ ]:
# Distribución del objetivo: las clases están desbalanceadas.
display(df["Churn"].value_counts())
display(df["Churn"].value_counts(normalize=True).round(3))

## 3. Evidencia para elegir variables

- `customerID` es un identificador: no entra.
- `TotalCharges` es casi antigüedad por cargo mensual, y sus vacíos son exactamente los clientes con `tenure = 0`.
- `gender` no cambia la tasa de fuga y es un atributo sensible.

In [ ]:
display(df[["tenure", "MonthlyCharges", "TotalCharges"]].corr().round(3))
print("tenure de los TotalCharges vacíos:", df.loc[df["TotalCharges"].isna(), "tenure"].unique())
display(pd.crosstab(df["gender"], df["Churn"], normalize="index").round(3))

In [ ]:
# Variables con más relación con la fuga
for col in ["Contract", "InternetService", "PaymentMethod", "TechSupport"]:
    display(pd.crosstab(df[col], df["Churn"], normalize="index").round(3))

## 4. Entrenar y evaluar con `train.py`

In [ ]:
!python train.py

In [ ]:
import json
print(json.dumps(json.load(open("metrics.json")), indent=2, ensure_ascii=False))

## 5. Pruebas mínimas antes de Docker

In [ ]:
!python -m pytest -q

## 6. Probar Streamlit en Colab

In [ ]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true > /content/streamlit.log 2>&1 &
!sleep 8 && cat /content/streamlit.log

El túnel muestra una URL temporal. localtunnel pide una contraseña: es la IP pública de esta máquina de Colab, que imprime la primera línea.

Si la red bloquea el túnel, basta con que el log anterior diga que Streamlit inició.

In [ ]:
!curl -s https://loca.lt/mytunnelpassword && echo
!npx --yes localtunnel --port 8501

## 7. Reiniciar Streamlit sin reentrenar

In [ ]:
!pkill -f "streamlit run" ; sleep 2
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true > /content/streamlit.log 2>&1 &
!sleep 8 && curl -s localhost:8501/_stcore/health && echo && cat /content/streamlit.log